In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
folder_of_data = "/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/"

In [3]:
!pip3 install matplotlib
!pip3 install pandas
!pip3 install h5py
!pip3 install scipy numba cython multiprocess boost==1.74

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement boost==1.74 (from versions: 0.1)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for boost==1.74


In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

In [5]:
from src.util import find_file_recursive
from src.processing import process_session

In [6]:
# review files of path saved in folder_of_data variables
# not all mice have full data, still working on incomplete mice
# This will help you select which csv in the path to use
all_reach_states = find_file_recursive(folder_of_data, "per_reach_state.csv")

📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1
  ❌ 'per_reach_state.csv' not found
  📂 Found 21 subdirectories
  📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210421_results
    ❌ 'per_reach_state.csv' not found
    📂 Found 1 subdirectories
    📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210421_results/210421_rep1
      ❌ 'per_reach_state.csv' not found
      📂 Found 4 subdirectories
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210421_results/210421_rep1/begin_reach
        ✅ Found 'per_reach_state.csv'
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210421_results/210421_rep1/post_reach
        ✅ Found 'per_reach_state.csv'
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210421_results/210421_rep1/mid_reach
        ✅ Foun

In [7]:
def extract_stats_to_df(stats_dict, **kwargs):
    """Extract stats dict to a flat dataframe row, with optional extra columns (e.g. stim=0)."""
    row = {
        'energy_wins':            stats_dict['energy_wins'],
        'firing_wins':            stats_dict['firing_wins'],
        'ties':                   stats_dict['ties'],
        'n_sessions':             stats_dict['n_sessions'],
        'energy_win_pct':         stats_dict['energy_win_pct'],
        'mean_energy_distance':   stats_dict['mean_energy_distance'],
        'mean_firing_distance':   stats_dict['mean_firing_distance'],
        'std_energy_distance':    stats_dict['std_energy_distance'],
        'std_firing_distance':    stats_dict['std_firing_distance'],
        'mean_covariance':        stats_dict['mean_covariance'],
        'mean_correlation':       stats_dict['mean_correlation'],
        'energy_min_wins':        stats_dict['extremum_wins']['energy_min'],
        'energy_max_wins':        stats_dict['extremum_wins']['energy_max'],
        'firing_min_wins':        stats_dict['extremum_wins']['firing_min'],
        'firing_max_wins':        stats_dict['extremum_wins']['firing_max'],
        **kwargs
    }
    return pd.DataFrame([row])

In [8]:

import os
import multiprocessing as mp
from src.processing import process_session
from multiprocessing import Pool

matlab_file = "/data001/projects/enserrog/AbigailData/energy_over_time/"


for i in range(1, 2):
    session_data = {}
    
    for session in all_reach_states:
    
    
        rep = session.split("_results/")[1].split("/")[0].split("_")[1][3:]
        
        if not (("full_reach" in session) and (rep == str(i))):
                continue
        
        print(session)
        session_id = session.split('_results')[0][-6:]
        print(f"Session_id: {session_id}")
    
        df = pd.read_csv(session)
    
        session_data[session_id] = df
    
    
    
    # Build all (stim, session) task arguments
    # window = [375, 425]
    window = [390, 410]
    tasks = [
        (stim, session, session_data[session], window)
        for stim in range(0, 3)
        for session in session_data.keys()
    ]
    
    stim_sessions_extrema = {}
    with Pool() as pool:
        for stim, session, data_frame in pool.imap_unordered(process_session, tasks):
            print(f"← Received: stim={stim}, session={session}")
            stim_sessions_extrema.setdefault(stim, {})[session] = data_frame
    
    print("All done!")
    #
    
    for stim in range(0,3):
        closest_list = []
        closest_point = [0, 0, 0, 0]
        for session in stim_sessions_extrema[stim].keys():
            max_velocity_idx = stim_sessions_extrema[stim][session]['velocity'][1]
            max_acceleration_idx = stim_sessions_extrema[stim][session]['acceleration'][1]
    
            # close_set = [
            #     np.abs(max_velocity_idx - stim_sessions_extrema[stim][session]['energy'][0]),
            #     np.abs(max_velocity_idx - stim_sessions_extrema[stim][session]['energy'][1]),
            #     np.abs(max_velocity_idx - stim_sessions_extrema[stim][session]['firing_rate'][0]),
            #     np.abs(max_velocity_idx - stim_sessions_extrema[stim][session]['firing_rate'][1])
            # ]

            close_set = [
                np.abs(max_acceleration_idx - stim_sessions_extrema[stim][session]['energy'][0]),
                np.abs(max_acceleration_idx - stim_sessions_extrema[stim][session]['energy'][1]),
                np.abs(max_acceleration_idx - stim_sessions_extrema[stim][session]['firing_rate'][0]),
                np.abs(max_acceleration_idx - stim_sessions_extrema[stim][session]['firing_rate'][1])
            ]
            
            idx_closest = np.argmin(
                close_set
            )
    
            closest_list += [close_set]
            
            closest_point[idx_closest] += 1
        print(closest_point)
        print(np.array(closest_list))
    #
    
    from src.arbitration import within_session_test_with_plots
    results = within_session_test_with_plots(
        stim_sessions_extrema, 
        output_dir=f'./Arbitration/accel_results/within_session_test_rep_10ms_accel_w390_410_MCH/{i}',
        verbose=True
    )
    
    df = pd.concat([
        extract_stats_to_df(results['by_stimulus'][stim], stim=stim, rep=i)
        for stim in results['by_stimulus'].keys()
    ], ignore_index=True)

    df.to_csv(f'./Arbitration/accel_results/within_session_test_rep_10ms_accel_w390_410_MCH/{i}/results.csv')


/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210421_results/210421_rep1/full_reach/per_reach_state.csv
Session_id: 210421
/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210422_results/210422_rep1/full_reach/per_reach_state.csv
Session_id: 210422
/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210423_results/210423_rep1/full_reach/per_reach_state.csv
Session_id: 210423
/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210425_results/210425_rep1/full_reach/per_reach_state.csv
Session_id: 210425
/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210511_results/210511_rep1/full_reach/per_reach_state.csv
Session_id: 210511
/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210512_results/210512_rep1/full_reach/per_reach_state.csv
Session_id: 210512
/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomp_1/210514_results/210514_rep1/full_re